# Course Bootstrap

[Open in Colab](https://colab.research.google.com/github/henry753951/basketball-hackathon-course/blob/main/init_colab_en.ipynb)

[English](init_colab_en.ipynb) | [Japanese](init_colab_ja.ipynb) | [Traditional Chinese](init_colab.ipynb)

## Set Up the Course Project in Google Drive

### Colab Runtime Settings

Before running this notebook, open **Runtime → Change runtime type** in Colab and select
**T4 GPU** as the hardware accelerator. Another NVIDIA GPU is also suitable. **Do not select TPU**:
the Ultralytics/PyTorch workflow used in this course does not directly support it and would run on the CPU instead.

1. Detect the runtime and mount Google Drive.
2. Select a synchronization mode:
   - `fresh_install`: reinstall the entire course repository.
   - `update_preserve_assets`: update the course files while preserving student additions in `assets/`.
3. Download the latest repository snapshot from GitHub and synchronize it to Google Drive.
4. Check the course folder and install any required Colab packages.
5. Verify all three Day 4 models and confirm that `assets/results/` is writable in Google Drive.

> The update replaces notebooks and code in `src/`. Save a separate copy of any course files you have edited.


In [1]:
import os
os.environ["BASKETBALL_COURSE_LANGUAGE"] = "en"

from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    IN_COLAB = False

COURSE_TITLE = "Python Basketball Analytics"
DRIVE_MOUNTED = False

if IN_COLAB and drive is not None:
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("This Colab runtime cannot mount Google Drive; using local /content storage.")

if DRIVE_MOUNTED:
    ROOT = Path("/content/drive/MyDrive/basketball_hackathon")
    TARGET_DIR = ROOT / "course"
elif IN_COLAB:
    ROOT = Path("/content/basketball_hackathon")
    TARGET_DIR = ROOT / "course"
else:
    ROOT = Path.cwd().resolve()
    TARGET_DIR = ROOT
    print("Local validation: using the current course repository.")

ROOT.mkdir(parents=True, exist_ok=True)

print(COURSE_TITLE)
print("Drive mounted:", DRIVE_MOUNTED)
print("Working directory:", ROOT)
print("Course directory:", TARGET_DIR)


Local validation: using the current course repository.
Python Basketball Analytics
Drive mounted: False
Working directory: /Users/henry753951/Developer/sandbox/basketball-hackathon-course
Course directory: /Users/henry753951/Developer/sandbox/basketball-hackathon-course


In [2]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/henry753951/basketball-hackathon-course.git"
REPO_BRANCH = "main"
SYNC_MODE = "update_preserve_assets"  # Or select "fresh_install".
PRESERVE_PATHS = ("assets",)

if IN_COLAB:
    LOCAL_CLONE_DIR = Path("/content/basketball_hackathon_course_tmp")
    if LOCAL_CLONE_DIR.exists():
        shutil.rmtree(LOCAL_CLONE_DIR)

    cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "-b",
        REPO_BRANCH,
        REPO_URL,
        str(LOCAL_CLONE_DIR),
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    LOCAL_CLONE_DIR = TARGET_DIR
    print("Local validation: skipping git clone.")

if str(LOCAL_CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_CLONE_DIR))

from src.course_setup import replace_tree_from_snapshot, rsync_tree  # noqa: E402

print("Source directory:", LOCAL_CLONE_DIR)
print("Sync mode:", SYNC_MODE)
if SYNC_MODE == "update_preserve_assets":
    print("Preserved paths:", ", ".join(PRESERVE_PATHS))
print("Assets are merged back; newer repository files take precedence for matching names. Notebooks and src are updated.")


Local validation: skipping git clone.
Source directory: /Users/henry753951/Developer/sandbox/basketball-hackathon-course
Sync mode: update_preserve_assets
Preserved paths: assets
Assets are merged back; newer repository files take precedence for matching names. Notebooks and src are updated.


In [3]:
import shutil

if IN_COLAB:
    if SYNC_MODE == "fresh_install":
        if TARGET_DIR.exists():
            shutil.rmtree(TARGET_DIR)
        rsync_tree(LOCAL_CLONE_DIR, TARGET_DIR)
    elif SYNC_MODE == "update_preserve_assets":
        replace_tree_from_snapshot(
            LOCAL_CLONE_DIR,
            TARGET_DIR,
            preserve_paths=PRESERVE_PATHS,
        )
    else:
        raise ValueError(f"Unsupported SYNC_MODE: {SYNC_MODE}")
else:
    print("Local validation: course files are already in this repository.")

print("Course files ready:", TARGET_DIR)


Local validation: course files are already in this repository.
Course files ready: /Users/henry753951/Developer/sandbox/basketball-hackathon-course


In [4]:
import subprocess
import sys

requirements = TARGET_DIR / "requirements.txt"
if IN_COLAB and requirements.exists():
    print("Checking Colab course packages...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
elif IN_COLAB:
    print("requirements.txt is missing. Check that the repository is complete.")
else:
    print("Local validation: packages are managed with uv sync; skipping pip install.")

print("\nCourse directory contents:")
for p in sorted(TARGET_DIR.iterdir()):
    print("-", p.name)


Local validation: packages are managed with uv sync; skipping pip install.

Course directory contents:
- .DS_Store
- .git
- .gitattributes
- .gitignore
- .ruff_cache
- .tmp
- .venv
- .vscode
- PROJECT_PLAN.md
- README.md
- assets
- day1
- day2
- day3
- day4
- day5
- init_colab.ipynb
- init_colab_en.ipynb
- init_colab_ja.ipynb
- output
- pyproject.toml
- python_basketball_hackathon_course_material.tex
- python_basketball_hackathon_course_material_en.tex
- python_basketball_hackathon_course_material_ja.tex
- requirements.txt
- scripts
- slides
- src
- test
- uv.lock
- workshop_schedule_updated.tex
- workshop_schedule_updated_en.tex
- workshop_schedule_updated_ja.tex


In [5]:
from src.course_setup import prepare_day4_workspace

DAY4_MODELS, DAY4_RESULTS = prepare_day4_workspace(TARGET_DIR)
print("\nDay 4 model checks:")
for model_name, model_path in DAY4_MODELS.items():
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"- {model_name}: {model_path} ({size_mb:.1f} MB)")
print("Day 4 results directory:", DAY4_RESULTS)
print("✅ Day 4 models are complete; results write check passed.")



Day 4 model checks:
- ball_rim: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/ball_rimV8.pt (6.0 MB)
- shot_detection: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/shot_detection.pt (5.9 MB)
- yolo_pose: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/pose/yolov8n-pose.pt (6.5 MB)
Day 4 results directory: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results
✅ Day 4 models are complete; results write check passed.


In [6]:
if IN_COLAB:
    # Google Drive path:
    drive_path = Path("/content/drive/MyDrive/basketball_hackathon")
    if drive_path.exists():
        print(f"Course directory created at {drive_path}")
        print("https://drive.google.com")
    